# GraphKV 第二阶段：ORION 错误归因与图依赖消融

第一阶段已实际通过单卡、位置、切片、不变性与小型 Llama dense mask 数值检查，但 ORION 被错误回答为 silver。这里区分 **prompt 格式、实体绑定、图边对表示的影响、query 直接读取 source**。

这是待在本地运行的研究 Notebook，没有预填推理结果。保留原模型、v5 模块和五节点文本；不修改库。默认物理 GPU 3，BF16，SDPA。重启内核后从头运行；同时运行旧 Notebook 会额外占用模型显存。

实验包括：原问题复现、legacy/native prompt 对比、逐项目与合并提问、正确边/无边/删边/换边、all/targets-only/sources-only query、顺序输入基线、候选答案 log probability、缓存变化局部性及 query cache 排列稳定性。

注意：换边保留原文本，是故意制造图与文本冲突的压力测试，不会自动把 ORION 的正确语义答案改为 silver。targets-only 是诊断条件，不是论文默认推理设置。此单例不能用于估计基准准确率。

依据：[v5](https://github.com/zmsnhcy/GraphKV_refactor_v5)、[论文 v2 §3.1](https://arxiv.org/html/2506.07334v2#S3.SS1)。本次优先读取本地 tokenizer 配置；远程配置读取未成功，因此不预先断言哪种模板是训练时正确模板。


## 1. 环境与单卡加载
沿用实际测试的路径与 GPU，先核对再运行。


In [1]:
import os, sys
from pathlib import Path
GPU_ID = "3"
REPO_DIR = Path("/home/lyh/hcy_work/GraphKV_refactor_v5")  # 改为含两个 graph_kv 模块的目录
MODEL_PATH = Path("/home/lyh/hcy_work/GraphKV/models/Tulu3-Block-FT")
BACKEND = "sdpa"  # 默认不依赖 flash-attn；也可用 eager / flash_attention_2
MAX_NEW_TOKENS = 64
if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    raise RuntimeError("CUDA 已初始化，请重启内核，再从本格运行。")
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# GitHub 仓库外层可能还有一层同名目录。
if not (REPO_DIR / "graph_kv_cache.py").is_file():
    nested = REPO_DIR / "GraphKV_refactor_v5"
    if (nested / "graph_kv_cache.py").is_file():
        REPO_DIR = nested
assert (REPO_DIR / "graph_kv_cache.py").is_file(), f"请修改 REPO_DIR: {REPO_DIR}"
assert (MODEL_PATH / "config.json").is_file(), f"找不到本地模型: {MODEL_PATH}"
sys.path.insert(0, str(REPO_DIR.resolve()))
print("模块目录:", REPO_DIR, "模型目录:", MODEL_PATH, "物理 GPU:", GPU_ID, sep="\n")


模块目录:
/home/lyh/hcy_work/GraphKV_refactor_v5
模型目录:
/home/lyh/hcy_work/GraphKV/models/Tulu3-Block-FT
物理 GPU:
3


In [2]:
import json, inspect, gc, time
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from graph_kv_cache import GraphKVCache
from graph_kv_adapter import greedy_generate_graphkv, graphkv_next_logits, build_graphkv_from_ids
from IPython.display import display, HTML
assert transformers.__version__ == "4.50.0", "v5 adapter 要求 transformers==4.50.0，请选择原有兼容内核。"
assert torch.cuda.is_available(), "当前内核没有可用 CUDA"
assert torch.cuda.device_count() == 1, "单卡屏蔽未生效，请重启内核"
assert torch.cuda.is_bf16_supported(), "此测试默认使用 BF16"
print("torch:", torch.__version__, "transformers:", transformers.__version__)
print("实际导入:", inspect.getfile(GraphKVCache))
print("GPU:", torch.cuda.get_device_name(0))
def memory(label):
    free, total = torch.cuda.mem_get_info(0)
    print(f"{label}: free={free/2**30:.2f}/{total/2**30:.2f} GiB; "
          f"allocated={torch.cuda.memory_allocated()/2**30:.2f} GiB; "
          f"peak={torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
memory("加载前")


/home/lyh/miniconda3/envs/graphkv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.5.1+cu121 transformers: 4.50.0
实际导入: /home/lyh/hcy_work/GraphKV_refactor_v5/graph_kv_cache.py
GPU: NVIDIA GeForce RTX 4090
加载前: free=23.26/23.64 GiB; allocated=0.00 GiB; peak=0.00 GiB


In [3]:
SOURCE_IDS = ["4", "5"]
TARGET_IDS = ["1", "2", "3"]
EDGES = [("4", "1"), ("4", "3"), ("5", "2"), ("5", "3")]
source_texts = {
    "4": "[Document 4] In this fictional dataset, ALPHA supplies copper. Its code is AX17.\n",
    "5": "[Document 5] In this fictional dataset, BETA supplies silver. Its code is BY29.\n",
}
target_texts = {
    "1": "[Document 1] Project ORION uses the material supplied by ALPHA.\n",
    "2": "[Document 2] Project LYRA uses the material supplied by BETA.\n",
    "3": "[Document 3] Project NOVA combines the materials supplied by ALPHA and BETA.\n",
}
PREFIX = "<|user|>\nAnswer using the following documents. Give a short factual answer.\n"
QUERY = "\nQuestion: What material does ORION use, what material does LYRA use, and which two materials does NOVA combine?\n<|assistant|>\n"
assert set(source_texts).isdisjoint(target_texts)
assert all(s in source_texts and t in target_texts for s,t in EDGES)
print("边:", EDGES)
for n, text in {**source_texts, **target_texts}.items():
    print(n, "source" if n in SOURCE_IDS else "target", text)
print("target <- sources:", {t:[s for s,d in EDGES if d==t] for t in TARGET_IDS})
display(HTML("<pre>4 ─────► 1\n│\n└──────► 3 ◄──────┐\n                   │\n5 ─────► 2         │\n└──────────────────┘</pre>"))


边: [('4', '1'), ('4', '3'), ('5', '2'), ('5', '3')]
4 source [Document 4] In this fictional dataset, ALPHA supplies copper. Its code is AX17.

5 source [Document 5] In this fictional dataset, BETA supplies silver. Its code is BY29.

1 target [Document 1] Project ORION uses the material supplied by ALPHA.

2 target [Document 2] Project LYRA uses the material supplied by BETA.

3 target [Document 3] Project NOVA combines the materials supplied by ALPHA and BETA.

target <- sources: {'1': ['4'], '2': ['5'], '3': ['4', '5']}


In [4]:
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
config = AutoConfig.from_pretrained(str(MODEL_PATH), local_files_only=True)
assert config.model_type == "llama"
rope = config.rope_scaling or {}
assert rope.get("rope_type", rope.get("type", "default")) in ("default", "llama3", "linear")
def tokenize(text):
    return tokenizer(text, return_tensors="pt", add_special_tokens=False,
                     truncation=False).input_ids
node_inputs = {n:tokenize(t) for n,t in {**source_texts, **target_texts}.items()}
prefix_ids, query_ids = tokenize(PREFIX), tokenize(QUERY)
lengths = {n:int(x.shape[1]) for n,x in node_inputs.items()}
P = int(prefix_ids.shape[1])
L = max(lengths.values())
Q = int(query_ids.shape[1])
assert P + 2*L + Q + MAX_NEW_TOKENS <= config.max_position_embeddings
print("P, L, Q =", P, L, Q)
print("node | kind | tokens | logical range [start,end)")
for n in SOURCE_IDS + TARGET_IDS:
    start = P if n in SOURCE_IDS else P+L
    print(n, "source" if n in SOURCE_IDS else "target", lengths[n], (start,start+lengths[n]))
print("query logical:", (P+2*L, P+2*L+Q))
print("query physical start:", P+sum(lengths.values()))
print("Tulu markers:", {x:tokenizer.encode(x, add_special_tokens=False) for x in ["<|user|>","<|assistant|>"]})


P, L, Q = 17 21 31
node | kind | tokens | logical range [start,end)
4 source 20 (17, 37)
5 source 21 (17, 38)
1 target 15 (38, 53)
2 target 16 (38, 54)
3 target 18 (38, 56)
query logical: (59, 90)
query physical start: 107
Tulu markers: {'<|user|>': [27, 91, 882, 91, 29], '<|assistant|>': [27, 91, 78191, 91, 29]}


In [5]:
torch.manual_seed(2026)
torch.cuda.reset_peak_memory_stats()
model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_PATH), local_files_only=True, torch_dtype=torch.bfloat16,
    device_map={"": 0}, low_cpu_mem_usage=True, attn_implementation=BACKEND,
).eval()
devices = {str(p.device) for p in model.parameters()}
assert devices == {"cuda:0"}, devices
assert next(model.parameters()).dtype == torch.bfloat16
device = model.get_input_embeddings().weight.device
print("所有参数所在设备:", devices, "backend:", model.config._attn_implementation)
memory("模型加载后")


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.24it/s]

所有参数所在设备: {'cuda:0'} backend: sdpa
模型加载后: free=8.30/23.64 GiB; allocated=14.96 GiB; peak=14.96 GiB


## 2. 本地 tokenizer、模板与停止符审计
marker 被拆成多个 token 并不足以证明格式错误。打印本地 `chat_template`、特殊 token、EOS 和用模板渲染的实际文本。不把 EOS 配置静默替换为猜测的值。


In [6]:
from graph_kv_adapter import build_graphkv_cache
import re, hashlib, math
audit = {
    "chat_template":tokenizer.chat_template,
    "special_tokens_map":tokenizer.special_tokens_map,
    "tokenizer_eos_token":tokenizer.eos_token,
    "tokenizer_eos_token_id":tokenizer.eos_token_id,
    "model_eos_token_id":model.generation_config.eos_token_id,
    "model_bos_token_id":model.generation_config.bos_token_id,
}
print(json.dumps(audit,ensure_ascii=False,indent=2,default=str))
for name in ["tokenizer_config.json","special_tokens_map.json","generation_config.json"]:
    path=MODEL_PATH/name
    if path.exists():
        obj=json.loads(path.read_text(encoding="utf-8"))
        print(name, {k:obj[k] for k in ["chat_template","bos_token","eos_token","eos_token_id","bos_token_id"] if k in obj})
INSTRUCTION = "Answer using the following documents. Give a short factual answer.\n"
MARKER = "__GRAPHKV_DOCUMENT_INSERTION_6DB831__"
def native_parts(question):
    rendered=tokenizer.apply_chat_template(
        [{"role":"user","content":INSTRUCTION+MARKER+"\nQuestion: "+question+"\n"}],
        tokenize=False,add_generation_prompt=True)
    assert rendered.count(MARKER)==1
    pre,suffix=rendered.split(MARKER)
    return pre,suffix,rendered
formats={"legacy":{"prefix":PREFIX,"native":False}}
if tokenizer.chat_template:
    pre,suffix,rendered=native_parts("What material does ORION use?")
    formats["native"]={"prefix":pre,"native":True}
    print("native rendered repr:",repr(rendered))
else:
    print("本地无 chat_template：跳过 native，不猜测模板。")
def question_suffix(fmt, question):
    if formats[fmt]["native"]:
        pre,suffix,_=native_parts(question)
        assert pre==formats[fmt]["prefix"]
        return suffix
    return "\nQuestion: "+question+"\n<|assistant|>\n"
for f in formats:
    print(f,"prefix:",repr(formats[f]["prefix"]),"suffix:",repr(question_suffix(f,"What material does ORION use?")))


{
  "chat_template": "{% for message in messages %}{% if message['role'] == 'system' %}{{ '<|system|>\n' + message['content'] + '\n' }}{% elif message['role'] == 'user' %}{{ '<|user|>\n' + message['content'] + '\n' }}{% elif message['role'] == 'assistant' %}{% if not loop.last %}{{ '<|assistant|>\n'  + message['content'] + eos_token + '\n' }}{% else %}{{ '<|assistant|>\n'  + message['content'] + eos_token }}{% endif %}{% endif %}{% if loop.last and add_generation_prompt %}{{ '<|assistant|>\n' }}{% endif %}{% endfor %}",
  "special_tokens_map": {
    "bos_token": "<|begin_of_text|>",
    "eos_token": "<|end_of_text|>",
    "pad_token": "<|end_of_text|>"
  },
  "tokenizer_eos_token": "<|end_of_text|>",
  "tokenizer_eos_token_id": 128001,
  "model_eos_token_id": 128001,
  "model_bos_token_id": 128000
}
tokenizer_config.json {'chat_template': "{% for message in messages %}{% if message['role'] == 'system' %}{{ '<|system|>\n' + message['content'] + '\n' }}{% elif message['role'] == 'user' %

## 3. 评价规则与实验计划
每个项目单独询问，让回答足够短，记录完整 raw tokens、EOS、截断、用时。材料识别只作粗粒度自动检查：copper 独有对应 ORION，silver 独有对应 LYRA，两者都出现对应 NOVA；否定、引用或复杂解释仍需人工复核。

另外对 ORION/LYRA 固定选择题式 prompt，对同一 query 分别计算 ` copper`、` silver` 续写的 teacher-forced log probability（包含全部候选 token），输出分差。候选分数对措辞和 token 长度敏感，仅比较同一 prompt 中的两候选，不替代自由回答评价。


In [7]:
questions={
 "ORION":"What material does ORION use? Answer with the material name only.",
 "LYRA":"What material does LYRA use? Answer with the material name only.",
 "NOVA":"Which two materials does NOVA combine? Answer with the material names only.",
}
COMBINED = "What material does ORION use, what material does LYRA use, and which two materials does NOVA combine?"
expected={"ORION":{"copper"},"LYRA":{"silver"},"NOVA":{"copper","silver"}}
graphs={
 "correct":EDGES,
 "no_edges":[],
 "drop_4_to_1":[e for e in EDGES if e!=("4","1")],
 "swap_1_2":[("5","1"),("4","2"),("4","3"),("5","3")],
}
views={
 "all":(SOURCE_IDS,TARGET_IDS),
 "targets_only":([],TARGET_IDS),
 "sources_only":(SOURCE_IDS,[]),
}
def material_check(project,text):
    found={w for w in ("copper","silver") if re.search(r"\b"+w+r"\b",text.lower())}
    return {"found":sorted(found),"rough_pass":found==expected[project]}
records=[]; score_records=[]; locality=[]; permutation=[]
RUN_CANDIDATE_SCORES = True
print("自由生成计划:",len(formats)*len(graphs)*len(views)*len(questions),"次，另有复现与基线。")


自由生成计划: 72 次，另有复现与基线。


In [8]:
def selected_cache_hash(g):
    h=hashlib.sha256()
    for n in [*SOURCE_IDS,*TARGET_IDS]:
        for xs in (g.nodes[n].key_cache,g.nodes[n].value_cache):
            for x in xs: h.update(x.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes())
    return h.hexdigest()
@torch.inference_mode()
def run_graph(g,fmt,graph_name,view,project,question=None):
    ss,tt=views[view]
    text=question_suffix(fmt,question or questions[project])
    torch.cuda.synchronize(); begin=time.perf_counter()
    r=greedy_generate_graphkv(model=model,tokenizer=tokenizer,graph_cache=g,
        query=text,source_ids=ss,target_ids=tt,max_new_tokens=MAX_NEW_TOKENS,return_details=True)
    torch.cuda.synchronize()
    row={"format":fmt,"graph":graph_name,"view":view,"project":project,
         "query":text,"answer":r.text,"token_ids":r.token_ids,
         "raw_text":tokenizer.decode(r.token_ids,skip_special_tokens=False),
         "eos":r.stopped_on_eos,"hit_limit":len(r.token_ids)==MAX_NEW_TOKENS and not r.stopped_on_eos,
         "seconds":time.perf_counter()-begin}
    if project in expected: row.update(material_check(project,r.text))
    records.append(row)
    print(fmt,graph_name,view,project,repr(r.text),"rough=",row.get("rough_pass"),"limit=",row["hit_limit"])
    return row
@torch.inference_mode()
def continuation_logp(g,query_text,candidate,ss,tt):
    q=tokenize(query_text).to(device); a=tokenize(candidate).to(device)
    # 明确独立 tokenize 后拼接；与候选 raw text 合并分词可能不同，记录候选 IDs。
    both=torch.cat([q,a],dim=1)
    past,pos,slots=g.prepare_query(both,source_ids=ss,target_ids=tt)
    mask=torch.ones((1,int(slots[-1])+1),dtype=torch.long,device=device)
    out=model(both,past_key_values=past,position_ids=pos,cache_position=slots,
              attention_mask=mask,use_cache=True,return_dict=True)
    start=q.shape[1]-1
    lp=out.logits[:,start:start+a.shape[1]].float().log_softmax(-1)
    vals=lp.gather(-1,a.unsqueeze(-1)).squeeze(-1)[0]
    return {"sum_logp":vals.sum().item(),"mean_logp":vals.mean().item(),
            "token_ids":a[0].tolist(),"per_token_logp":vals.tolist()}
def score_pair(g,fmt,name,view,project):
    ss,tt=views[view]
    prompt=question_suffix(fmt,f"What material does {project} use? Choose exactly one: copper or silver. Answer:")
    a=continuation_logp(g,prompt," copper",ss,tt)
    b=continuation_logp(g,prompt," silver",ss,tt)
    row={"format":fmt,"graph":name,"view":view,"project":project,"query":prompt,
         "copper":a,"silver":b,"copper_minus_silver":a["sum_logp"]-b["sum_logp"]}
    score_records.append(row)
    print("  score",project,"logP(copper)-logP(silver)=",round(row["copper_minus_silver"],4))
def cpu_snapshot(g):
    return {n:{attr:[x.detach().cpu().clone() for x in getattr(g.nodes[n],attr)]
               for attr in ["key_cache","value_cache"]} for n in SOURCE_IDS+TARGET_IDS}
def delta_node(ref,g,n):
    maximum=0.0; total=0.0; count=0
    for attr in ["key_cache","value_cache"]:
        for a,b in zip(ref[n][attr],getattr(g.nodes[n],attr)):
            d=(a.float()-b.detach().cpu().float()).abs()
            maximum=max(maximum,d.max().item()); total+=d.sum().item(); count+=d.numel()
    return {"max_abs":maximum,"mean_abs":total/count}


## 4. 原测试复现、完整图消融与缓存局部性
按格式和图逐个构建，释放后再构建下一个，避免累计 GPU 缓存。删 4→1 时只有 target 1 的表示有理由改变；source 4/5 和 target 2/3 应保持完全相同。换边时只改变 target 1/2，target 3 应保持相同。

无边基线保留相同 P/L 与 target 位置区间，控制位置因素；它不是把全部节点移到 source 区间的另一种 PCW 基线。sources-only 仍沿用完整图的 query 逻辑位置，不能把其语义分数当成公平的完整问答准确率。


In [9]:
for fmt in formats:
    reference=None
    for name,edges in graphs.items():
        gc.collect(); torch.cuda.empty_cache()
        g=build_graphkv_cache(model=model,tokenizer=tokenizer,
            source_texts=source_texts,target_texts=target_texts,edges=edges,
            prefix=formats[fmt]["prefix"])
        initial_hash=selected_cache_hash(g)
        if name=="correct":
            reference=cpu_snapshot(g)
            run_graph(g,fmt,name,"all","COMBINED",COMBINED)
            # source/target 的物理顺序改变，逻辑 PE 不变。
            query=tokenize(question_suffix(fmt,questions["ORION"]))
            la=graphkv_next_logits(model=model,graph_cache=g,query_input_ids=query)
            lb=graphkv_next_logits(model=model,graph_cache=g,query_input_ids=query,
                source_ids=list(reversed(SOURCE_IDS)),target_ids=list(reversed(TARGET_IDS)))
            permutation.append({"format":fmt,"max_abs_logits":(la-lb).abs().max().item(),
                "argmax_a":la.argmax(-1).item(),"argmax_b":lb.argmax(-1).item()})
            print("排列诊断:",permutation[-1])
            del la,lb
        else:
            deltas={n:delta_node(reference,g,n) for n in SOURCE_IDS+TARGET_IDS}
            unaffected={"no_edges":SOURCE_IDS,"drop_4_to_1":SOURCE_IDS+["2","3"],
                        "swap_1_2":SOURCE_IDS+["3"]}[name]
            for n in unaffected:
                assert deltas[n]["max_abs"]==0.0, (fmt,name,n,deltas[n])
            locality.append({"format":fmt,"graph":name,"node_deltas":deltas,
                             "unaffected_pass":True})
            print("缓存变化:",deltas)
        for view in views:
            for project in questions:
                run_graph(g,fmt,name,view,project)
                if RUN_CANDIDATE_SCORES and project in ["ORION","LYRA"]:
                    score_pair(g,fmt,name,view,project)
        assert selected_cache_hash(g)==initial_hash,"推理修改了复用节点缓存"
        del g
    del reference
memory("消融完成")


legacy correct all COMBINED 'According to the provided documents:\n\n* ORION uses the material supplied by BETA, which is silver.\n* LYRA uses the material supplied by BETA, which is silver.\n* NOVA combines the materials supplied by ALPHA (copper) and BETA (silver).' rough= None limit= False
排列诊断: {'format': 'legacy', 'max_abs_logits': 0.3125, 'argmax_a': 34, 'argmax_b': 34}
legacy correct all ORION 'Copper.' rough= True limit= False
  score ORION logP(copper)-logP(silver)= -0.875
legacy correct all LYRA 'Copper.' rough= False limit= False
  score LYRA logP(copper)-logP(silver)= -2.4375
legacy correct all NOVA 'According to the provided documents, Project NOVA combines copper supplied by ALPHA and silver supplied by BETA.' rough= True limit= False
legacy correct targets_only ORION 'According to the documents, ORION uses the material supplied by BETA, which is the material supplied by BETA.' rough= False limit= False
  score ORION logP(copper)-logP(silver)= -2.25
legacy correct targets

## 5. 顺序输入基线（保持格式与问题措辞）
一条序列 `[prefix,documents,query]`，普通连续位置。用 v5 的 greedy 生成器读取该序列：将连续编码的文档前缀注册为唯一 source，其位置从 0 开始，query 从文档长度开始，从而得到普通 sequential KV 解码，不额外调用不同的生成算法。

native 格式必须与一次性 `apply_chat_template` 渲染的完整文本完全一致。对两种文档顺序分别测试；这能帮助观察顺序与实体绑定敏感性，但不足以断言图机制优劣。


In [10]:
@torch.inference_mode()
def sequential_cache(prefix_text):
    ids=tokenize(prefix_text).to(device)
    slots=torch.arange(ids.shape[1],device=device)
    out=model(ids,position_ids=slots.unsqueeze(0),cache_position=slots,
              attention_mask=torch.ones_like(ids),use_cache=True,return_dict=True)
    g=GraphKVCache(source_position_start=0,chunk_span=int(ids.shape[1]))
    g.add_source("sequence",out.past_key_values,assume_positioned=True)
    return g
all_texts={**source_texts,**target_texts}
orders={"source_first":SOURCE_IDS+TARGET_IDS,"target_first":TARGET_IDS+SOURCE_IDS}
for fmt in formats:
    for order,ns in orders.items():
        docs="".join(all_texts[n] for n in ns)
        pre=formats[fmt]["prefix"]+docs
        if formats[fmt]["native"]:
            for question in [*questions.values(),COMBINED]:
                rendered=tokenizer.apply_chat_template(
                    [{"role":"user","content":INSTRUCTION+docs+"\nQuestion: "+question+"\n"}],
                    tokenize=False,add_generation_prompt=True)
                assert pre+question_suffix(fmt,question)==rendered
        g=sequential_cache(pre)
        for project,question in [*questions.items(),("COMBINED",COMBINED)]:
            suffix=question_suffix(fmt,question)
            r=greedy_generate_graphkv(model=model,tokenizer=tokenizer,graph_cache=g,
                query=suffix,source_ids=["sequence"],target_ids=[],
                max_new_tokens=MAX_NEW_TOKENS,return_details=True)
            row={"format":fmt,"graph":"sequential_"+order,"view":"all","project":project,
                 "query":suffix,"answer":r.text,"token_ids":r.token_ids,"eos":r.stopped_on_eos,
                 "hit_limit":len(r.token_ids)==MAX_NEW_TOKENS and not r.stopped_on_eos}
            if project in expected: row.update(material_check(project,r.text))
            records.append(row)
            print(fmt,order,project,repr(r.text),row.get("rough_pass"))
        del g
        gc.collect(); torch.cuda.empty_cache()


legacy source_first ORION 'Copper' True
legacy source_first LYRA 'According to Document 2, Project LYRA uses the material supplied by BETA, which is silver.' True
legacy source_first NOVA 'Copper and silver.' True
legacy source_first COMBINED 'According to the provided documents, Project ORION uses copper, Project LYRA uses silver, and Project NOVA combines copper and silver.' None
legacy target_first ORION 'Copper.' True
legacy target_first LYRA 'Silver.' True
legacy target_first NOVA 'Copper and silver.' True
legacy target_first COMBINED 'According to the provided documents, Project ORION uses copper, Project LYRA uses silver, and Project NOVA combines copper and silver.' None
native source_first ORION 'Copper' True
native source_first LYRA 'According to Document 2, Project LYRA uses the material supplied by BETA, which is silver.' True
native source_first NOVA 'According to Document 3, Project NOVA combines copper and silver.' True
native source_first COMBINED 'According to the prov

## 6. 汇总与保存（不自动作因果结论）
- legacy 错、native 与顺序基线对：提示格式值得优先研究；仍需跨问题/样本验证。
- 正确图和无边结果不同：证明边能影响该设置下输出，未必提高正确率。
- 删边时无关节点不变、相关 target 改变：支持缓存传播符合依赖局部性。
- all 对、targets-only 错：可能依赖 source 直接读取，也可能因剥离 source 改变输入分布，不能简单说传播没有携带信息。
- 所有设置都误绑定 ORION：继续研究文本/分词/实体顺序与模型本身。
- BF16 下排列 logits 有微小差异可能源于浮点求和顺序，不能直接认定位置设计错误。

本 Notebook 未执行于制作环境；下面的实际 JSON/CSV 才是下一阶段分析依据。


In [11]:
import csv
output_dir=Path.cwd()/"graphkv_stage2_outputs"
output_dir.mkdir(exist_ok=True)
payload={"audit":audit,"settings":{"model_path":str(MODEL_PATH),"repo_dir":str(REPO_DIR),
    "physical_gpu":GPU_ID,"backend":BACKEND,"max_new_tokens":MAX_NEW_TOKENS,
    "torch":torch.__version__,"transformers":transformers.__version__},
    "source_texts":source_texts,"target_texts":target_texts,"graphs":graphs,
    "formats":formats,"records":records,"candidate_scores":score_records,
    "locality":locality,"permutation":permutation}
(output_dir/"graphkv_stage2_results.json").write_text(
    json.dumps(payload,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
fields=["format","graph","view","project","answer","rough_pass","eos","hit_limit"]
with (output_dir/"answers.csv").open("w",newline="",encoding="utf-8-sig") as f:
    writer=csv.DictWriter(f,fieldnames=fields,extrasaction="ignore"); writer.writeheader(); writer.writerows(records)
print("已保存:",output_dir)
print("format | graph | view | ORION / LYRA / NOVA rough check")
for fmt in formats:
    for name in [*graphs,*["sequential_"+o for o in orders]]:
        for view in (views if name in graphs else ["all"]):
            rows=[r for r in records if r["format"]==fmt and r["graph"]==name and r["view"]==view and r["project"] in expected]
            if rows: print(fmt,name,view,{r["project"]:r["rough_pass"] for r in rows})
print("请人工检查 COMBINED 输出和包含否定/额外解释的回答；rough_pass 不是最终准确率。")


已保存: /home/lyh/hcy_work/GraphKV_refactor_v5/graphkv_stage2_outputs
format | graph | view | ORION / LYRA / NOVA rough check
legacy correct all {'ORION': True, 'LYRA': False, 'NOVA': True}
legacy correct targets_only {'ORION': False, 'LYRA': False, 'NOVA': False}
legacy correct sources_only {'ORION': False, 'LYRA': False, 'NOVA': False}
legacy no_edges all {'ORION': True, 'LYRA': False, 'NOVA': False}
legacy no_edges targets_only {'ORION': False, 'LYRA': False, 'NOVA': False}
legacy no_edges sources_only {'ORION': False, 'LYRA': False, 'NOVA': False}
legacy drop_4_to_1 all {'ORION': True, 'LYRA': False, 'NOVA': True}
legacy drop_4_to_1 targets_only {'ORION': False, 'LYRA': False, 'NOVA': False}
legacy drop_4_to_1 sources_only {'ORION': False, 'LYRA': False, 'NOVA': False}
legacy swap_1_2 all {'ORION': True, 'LYRA': False, 'NOVA': False}
legacy swap_1_2 targets_only {'ORION': False, 'LYRA': False, 'NOVA': False}
legacy swap_1_2 sources_only {'ORION': False, 'LYRA': False, 'NOVA': False}
l